# FastViT Object Detection — MedPackID

Notebook Google Colab untuk deteksi objek kemasan obat dan pil menggunakan backbone FastViT, Feature Pyramid Network (FPN), dan Faster R-CNN. Ini **object detection** (kelas + bounding box), bukan image classification.

Sumber data: [MedPackID di Mendeley Data](https://data.mendeley.com/datasets/d23kndp3sv/1), versi 1. Halaman dataset menyebut 1.444 gambar, split training/validation/testing, anotasi bounding box teks, dan lisensi CC BY 4.0. Notebook mengambil arsip langsung dari Mendeley Data; ukuran unduhan sekitar 3 GB, jadi siapkan ruang disk dan waktu di runtime Colab.

**Input awal 320 px** dipilih sebagai eksperimen resolusi sekitar 300 px yang ramah GPU Colab. Ini bukan default deteksi yang diklaim paper: FastViT melaporkan evaluasi backbone deteksi pada crop 512×512 dan Mask R-CNN dengan backbone pra-latih ImageNet. FPN menggabungkan peta fitur lintas resolusi lewat lateral connections dan nearest-neighbor upsampling. Detail yang hilang saat gambar input diperkecil tidak bisa dipulihkan dengan upsampling. Training langsung memakai learning rate yang ditentukan sejak epoch pertama, tanpa warmup.


## 1. Runtime Colab

Pilih **Runtime → Change runtime type → T4 GPU** (atau GPU lain). Jalankan sel berurutan. Arsip 3.03 GB diunduh langsung dari Mendeley ke VM Colab sehingga tidak menggunakan kuota unduhan komputer Anda; ruang disk Colab tetap terpakai selama proses.


In [ ]:
!pip -q install timm


In [ ]:
import os, random, json, math, time
from pathlib import Path
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision.models.detection import FasterRCNN
from torchvision.models.detection.rpn import AnchorGenerator
from torchvision.ops import FeaturePyramidNetwork
from torchvision.transforms import functional as TF
from PIL import Image
import timm

assert torch.cuda.is_available(), "Aktifkan GPU di Runtime > Change runtime type."
DEVICE = torch.device("cuda")
print(torch.cuda.get_device_name(0), torch.__version__, timm.__version__)


## 2. Unduh dan ekstrak MedPackID dari Mendeley Data

Unduhan memakai URL publik yang digunakan tombol **Download All** pada halaman dataset. Request dan arsip 3.03 GB berjalan langsung di runtime Google Colab, bukan melalui koneksi internet komputer Anda. Sel ini memverifikasi SHA-256 yang diberikan sebelum ekstraksi dan juga membuka arsip ZIP di dalam paket utama (`images.zip`, `labels.zip`, `annotations_voc.zip`).


In [ ]:
import requests, zipfile, time, shutil, os, hashlib
from tqdm.auto import tqdm

DATASET_ID = "d23kndp3sv"
DATASET_VERSION = 1
DATA_DIR = Path("/content/MedPackID")
ZIP_PATH = Path("/content/MedPackID_v1.zip")
EXPECTED_SHA256 = "c47e1b5133765801805e41ce4c3a688126aeaaf257d6e0af7c151ff6ae0df835"
# This is the public Download All link shown on the Mendeley dataset page.
DOWNLOAD_URL = f"https://data.mendeley.com/public-api/zip/{DATASET_ID}/download/{DATASET_VERSION}"

# Stream directly from Mendeley to the Colab VM; the bytes do not pass through your computer.
if not ZIP_PATH.exists() or ZIP_PATH.stat().st_size == 0:
    with requests.get(DOWNLOAD_URL, stream=True, timeout=(90, 300)) as response:
        if response.status_code == 401:
            raise RuntimeError("Mendeley menolak URL unduhan publik (HTTP 401). Pastikan link Download All dataset masih bisa dibuka tanpa login.")
        response.raise_for_status()
        total = int(response.headers.get("content-length", 0))
        with open(ZIP_PATH, "wb") as f, tqdm(total=total or None, unit="B", unit_scale=True,
                                               desc="MedPackID dari Mendeley → Colab") as bar:
            for chunk in response.iter_content(chunk_size=1024*1024):
                if chunk:
                    f.write(chunk)
                    bar.update(len(chunk))

# Validate the downloaded Download All archive before extraction.
def sha256_file(path):
    digest = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(8 * 1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()

actual_sha256 = sha256_file(ZIP_PATH)
print("SHA-256 arsip Download All:", actual_sha256)
if actual_sha256.lower() != EXPECTED_SHA256.lower():
    raise RuntimeError(
        "Checksum arsip Download All tidak cocok. "
        f"Harapan: {EXPECTED_SHA256}; aktual: {actual_sha256}. "
        "Arsip tidak diekstrak."
    )
if not zipfile.is_zipfile(ZIP_PATH):
    ZIP_PATH.unlink(missing_ok=True)
    raise RuntimeError("Respons Mendeley bukan ZIP yang valid. Arsip dihapus; jalankan ulang sel unduh.")

DATA_DIR.mkdir(parents=True, exist_ok=True)
if not any(DATA_DIR.iterdir()):
    with zipfile.ZipFile(ZIP_PATH) as archive:
        bad_member = archive.testzip()
        if bad_member:
            raise RuntimeError(f"Arsip Download All rusak pada file: {bad_member}")
        archive.extractall(DATA_DIR)

# Download All includes separate images.zip, labels.zip and annotations_voc.zip.
# Extract nested ZIPs too so the dataset loader can see image and annotation folders.
while True:
    nested = [p for p in DATA_DIR.rglob("*.zip") if p.is_file()]
    if not nested:
        break
    extracted_any = False
    for archive_path in nested:
        marker = archive_path.with_suffix(archive_path.suffix + ".extracted")
        if marker.exists():
            continue
        with zipfile.ZipFile(archive_path) as archive:
            bad_member = archive.testzip()
            if bad_member:
                raise RuntimeError(f"Arsip rusak: {archive_path.name}, entri: {bad_member}")
            archive.extractall(archive_path.parent)
        marker.touch()
        extracted_any = True
        print("Diekstrak:", archive_path.name)
    if not extracted_any:
        break

print("Dataset siap:", DATA_DIR)
for path in list(DATA_DIR.rglob("*"))[:50]:
    if path.is_file() and path.suffix.lower() not in {".jpg", ".jpeg", ".png", ".bmp", ".webp"}:
        print(path.relative_to(DATA_DIR))


## 3. Dataset LabelImg (YOLO TXT atau Pascal VOC XML)

Halaman dataset mendeskripsikan bounding box sebagai anotasi teks. Loader mencoba format YOLO `.txt` (koordinat ternormalisasi) dan LabelImg Pascal VOC `.xml`; kelas dibaca dari `classes.txt`/`obj.names`/`data.yaml` bila tersedia, atau diindeks dari anotasi. Ia mencari folder split train, valid/val, dan test secara otomatis.


In [ ]:
import re, ast, xml.etree.ElementTree as ET

IMAGE_EXTS={".jpg", ".jpeg", ".png", ".bmp", ".webp"}
def split_folder(name_options):
    wanted={n.lower() for n in name_options}
    for p in DATA_DIR.rglob("*"):
        if p.is_dir() and p.name.lower() in wanted:
            # Prefer an image directory if split contains images/labels subfolders.
            if any(q.suffix.lower() in IMAGE_EXTS for q in p.rglob("*")):
                return p
    raise FileNotFoundError(f"Tidak menemukan split {name_options} di {DATA_DIR}")

def image_files(split_dir):
    return sorted(p for p in split_dir.rglob("*") if p.is_file() and p.suffix.lower() in IMAGE_EXTS)

def annotation_for(img, split_dir):
    candidates=[]
    parts=list(img.parts)
    for i,part in enumerate(parts):
        if part.lower()=="images":
            alt=parts.copy(); alt[i]="labels"
            candidates.append(Path(*alt).with_suffix(".txt"))
    candidates += [img.with_suffix(".txt"), img.with_suffix(".xml"), split_dir/"Annotations"/(img.stem+".xml"), split_dir/"annotations"/(img.stem+".xml")]
    for c in candidates:
        if c.exists(): return c
    # Mendeley distributes labels and VOC XML in separate archives.
    for suffix in (".xml", ".txt"):
        matches = list(DATA_DIR.rglob(img.stem + suffix))
        # Prefer an annotation from a folder with the same train/valid/test split.
        split_parts = {part.lower() for part in split_dir.parts}
        same_split = [m for m in matches if split_parts.intersection(part.lower() for part in m.parts)]
        if same_split: return same_split[0]
        if matches: return matches[0]
    return None

def parse_annotation(path, width, height):
    boxes=[]; class_ids=[]
    if path is None: return boxes, class_ids
    if path.suffix.lower()==".xml":
        root=ET.parse(path).getroot()
        for obj in root.findall("object"):
            name=(obj.findtext("name") or "class_0").strip()
            b=obj.find("bndbox")
            if b is None: continue
            x1=float(b.findtext("xmin",0)); y1=float(b.findtext("ymin",0)); x2=float(b.findtext("xmax",0)); y2=float(b.findtext("ymax",0))
            boxes.append([max(0,x1),max(0,y1),min(width,x2),min(height,y2)]); class_ids.append(name)
    else:
        for line in path.read_text(errors="ignore").splitlines():
            vals=line.split()
            if len(vals)<5: continue
            try: cid=int(float(vals[0])); a,b,c,d=map(float,vals[1:5])
            except ValueError: continue
            # LabelImg YOLO convention: class, normalized center-x/y, width/height.
            x1=(a-c/2)*width; y1=(b-d/2)*height; x2=(a+c/2)*width; y2=(b+d/2)*height
            boxes.append([max(0,x1),max(0,y1),min(width,x2),min(height,y2)]); class_ids.append(cid)
    valid=[(b,c) for b,c in zip(boxes,class_ids) if b[2]>b[0] and b[3]>b[1]]
    return [x[0] for x in valid],[x[1] for x in valid]

def read_names():
    for p in DATA_DIR.rglob("*"):
        if p.is_file() and p.name.lower() in {"classes.txt","obj.names"}:
            return [x.strip() for x in p.read_text().splitlines() if x.strip()]
        if p.is_file() and p.name.lower()=="data.yaml":
            text=p.read_text()
            m=re.search(r"(?m)^names:\s*(\[.*\])",text)
            if m:
                try: return list(ast.literal_eval(m.group(1)))
                except Exception: pass
            m=re.search(r"(?ms)^names:\s*\n((?:\s+-.*\n?)+)",text)
            if m: return [line.strip()[2:].strip().replace(chr(34), "").replace(chr(39), "") for line in m.group(1).splitlines()]
    return None

train_dir=split_folder(["train","training"])
valid_dir=split_folder(["valid","validation","val"])
test_dir=split_folder(["test","testing"])
CLASS_NAMES=read_names()

class MedPackDataset(Dataset):
    def __init__(self, split_dir, train=False, class_to_label=None):
        self.root=split_dir; self.train=train; self.paths=image_files(split_dir); self.class_to_label=class_to_label
        if not self.paths: raise RuntimeError(f"Tidak ada gambar di {split_dir}")
    def __len__(self): return len(self.paths)
    def __getitem__(self, idx):
        path=self.paths[idx]; image=Image.open(path).convert("RGB"); w,h=image.size
        ann=annotation_for(path,self.root); boxes,ids=parse_annotation(ann,w,h)
        labels=[]
        for cid in ids:
            if cid not in self.class_to_label: continue
            labels.append(self.class_to_label[cid])
        if len(labels)!=len(boxes):
            pairs=[(b,c) for b,c in zip(boxes,ids) if c in self.class_to_label]
            boxes=[p[0] for p in pairs]; labels=[self.class_to_label[p[1]] for p in pairs]
        boxes=torch.as_tensor(boxes,dtype=torch.float32).reshape(-1,4); labels=torch.as_tensor(labels,dtype=torch.int64)
        if self.train and random.random()<0.5:
            image=TF.hflip(image); boxes=boxes.clone()
            if len(boxes): boxes[:,[0,2]]=w-boxes[:,[2,0]]
        image=TF.to_tensor(image)
        target={"boxes":boxes,"labels":labels,"image_id":torch.tensor([idx]),
                "area":(boxes[:,2]-boxes[:,0])*(boxes[:,3]-boxes[:,1]),"iscrowd":torch.zeros(len(boxes),dtype=torch.int64)}
        return image,target

# Read all annotation class identifiers once to create a stable contiguous label map.
all_ids=set()
for split in [train_dir,valid_dir,test_dir]:
    for img_path in image_files(split):
        im=Image.open(img_path); ids=parse_annotation(annotation_for(img_path,split),*im.size)[1]; all_ids.update(ids)
if CLASS_NAMES is None:
    if all(isinstance(x,int) for x in all_ids):
        ordered=sorted(all_ids); CLASS_NAMES=[f"class_{i}" for i in ordered]
        class_to_label={cid:i+1 for i,cid in enumerate(ordered)}
    else:
        ordered=sorted(all_ids); CLASS_NAMES=[str(x) for x in ordered]
        class_to_label={cid:i+1 for i,cid in enumerate(ordered)}
else:
    class_to_label={i:i+1 for i in range(len(CLASS_NAMES))}
    class_to_label.update({name:i+1 for i,name in enumerate(CLASS_NAMES)})
CLASSES=CLASS_NAMES
NUM_CLASSES=len(CLASS_NAMES)+1
train_ds=MedPackDataset(train_dir,True,class_to_label)
valid_ds=MedPackDataset(valid_dir,False,class_to_label)
test_ds=MedPackDataset(test_dir,False,class_to_label)
print(f"Train {len(train_ds)} | valid {len(valid_ds)} | test {len(test_ds)} | classes {CLASS_NAMES}")
def collate_fn(batch): return tuple(zip(*batch))
BATCH_SIZE=4
train_loader=DataLoader(train_ds,batch_size=BATCH_SIZE,shuffle=True,num_workers=2,collate_fn=collate_fn,pin_memory=True)
valid_loader=DataLoader(valid_ds,batch_size=2,shuffle=False,num_workers=2,collate_fn=collate_fn,pin_memory=True)
test_loader=DataLoader(test_ds,batch_size=2,shuffle=False,num_workers=2,collate_fn=collate_fn,pin_memory=True)


## 4. Arsitektur FastViT + FPN + Faster R-CNN

Gunakan checkpoint FastViT ImageNet sebagai backbone pretrained. Paper menyediakan evaluasi detection berbasis Mask R-CNN, tetapi notebook ini memakai implementasi Faster R-CNN dari Torchvision agar instalasi lebih stabil di Colab. Head deteksi tetap perlu dilatih pada label MedPack ID. FPN meng-upsample peta fitur kasar dan menambahkan fitur lateral dari stage resolusi lebih tinggi.


In [ ]:
IMAGE_SIZE = 320  # eksperimen awal ~300 px; coba 384/512 bila objek kecil
BACKBONE_NAME = "fastvit_sa12.apple_in1k"

class FastViTFPN(nn.Module):
    def __init__(self, name=BACKBONE_NAME, pretrained=True, out_channels=192):
        super().__init__()
        self.body = timm.create_model(name, pretrained=pretrained, features_only=True, out_indices=(0,1,2,3))
        channels = self.body.feature_info.channels()
        self.fpn = FeaturePyramidNetwork(channels, out_channels, extra_blocks=None)
        self.out_channels = out_channels
    def forward(self, x):
        feats = self.body(x)
        return self.fpn({str(i): f for i,f in enumerate(feats)})

backbone = FastViTFPN()
anchor_sizes = ((16, 32), (32, 64), (64, 128), (128, 256))
aspect_ratios = ((0.5, 1.0, 2.0),) * len(anchor_sizes)
anchors = AnchorGenerator(anchor_sizes, aspect_ratios)
model = FasterRCNN(backbone, num_classes=NUM_CLASSES, rpn_anchor_generator=anchors,
                   min_size=IMAGE_SIZE, max_size=IMAGE_SIZE,
                   box_detections_per_img=100)
model.to(DEVICE)
print("FastViT feature channels:", backbone.body.feature_info.channels())


## 5. Training tanpa warmup

Training berlangsung 100 epoch. Learning rate ditetapkan langsung sejak epoch pertama, tanpa warmup. Scheduler `StepLR` menurunkan LR di tengah proses, tanpa fase warmup. Checkpoint terbaik dipilih berdasarkan mean validation loss (indikator praktis; evaluasi mAP sebaiknya ditambahkan untuk pelaporan final).


In [ ]:
EPOCHS = 100
LR = 2e-4
WEIGHT_DECAY = 1e-4
optimizer = torch.optim.AdamW([p for p in model.parameters() if p.requires_grad], lr=LR, weight_decay=WEIGHT_DECAY)
scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=max(1, EPOCHS*2//3), gamma=0.1)
SCALER = torch.amp.GradScaler("cuda", enabled=True)
OUT_DIR = Path("/content/fastvit_medpack_runs"); OUT_DIR.mkdir(exist_ok=True)

def run_epoch(loader, training):
    model.train(training); running=0.0; count=0
    for images, targets in loader:
        images=[im.to(DEVICE, non_blocking=True) for im in images]
        targets=[{k:v.to(DEVICE, non_blocking=True) for k,v in t.items()} for t in targets]
        if training:
            optimizer.zero_grad(set_to_none=True)
            with torch.autocast(device_type="cuda", dtype=torch.float16):
                losses = model(images, targets); loss = sum(losses.values())
            SCALER.scale(loss).backward(); SCALER.step(optimizer); SCALER.update()
        else:
            # Torchvision menghasilkan loss saat training=True; evaluasi loss tanpa grad.
            model.train();
            with torch.no_grad(), torch.autocast(device_type="cuda", dtype=torch.float16):
                losses = model(images, targets); loss = sum(losses.values())
            model.eval()
        running += float(loss.detach()); count += 1
    return running / max(count,1)

best_val = float("inf")
history=[]
for epoch in range(1, EPOCHS+1):
    t0=time.time()
    train_loss=run_epoch(train_loader, True)
    val_loss=run_epoch(valid_loader, False)
    scheduler.step()
    lr_now=optimizer.param_groups[0]["lr"]
    history.append({"epoch":epoch,"train_loss":train_loss,"val_loss":val_loss,"lr":lr_now})
    print(f"Epoch {epoch:02d}/{EPOCHS} | train {train_loss:.4f} | val {val_loss:.4f} | lr {lr_now:.2g} | {time.time()-t0:.0f}s")
    if val_loss < best_val:
        best_val=val_loss
        torch.save({"model":model.state_dict(),"classes":CLASSES,"image_size":IMAGE_SIZE,"epoch":epoch}, OUT_DIR/"best.pth")
with open(OUT_DIR/"history.json","w") as f: json.dump(history,f,indent=2)


## 6. Lihat kurva loss dan prediksi


In [ ]:
import matplotlib.pyplot as plt
xs=[h["epoch"] for h in history]
plt.plot(xs,[h["train_loss"] for h in history],label="train")
plt.plot(xs,[h["val_loss"] for h in history],label="valid")
plt.xlabel("Epoch"); plt.ylabel("Loss"); plt.legend(); plt.grid(); plt.show()


In [ ]:
checkpoint=torch.load(OUT_DIR/"best.pth", map_location=DEVICE, weights_only=False)
model.load_state_dict(checkpoint["model"]); model.eval()
img, _ = valid_ds[0]
with torch.no_grad(): pred=model([img.to(DEVICE)])[0]
from torchvision.utils import draw_bounding_boxes
from torchvision.transforms.functional import to_pil_image
keep=pred["scores"].cpu() > 0.35
canvas=draw_bounding_boxes((img*255).byte(), pred["boxes"].cpu()[keep],
    labels=[f"{CLASSES[i-1]} {s:.2f}" for i,s in zip(pred["labels"].cpu()[keep],pred["scores"].cpu()[keep])],
    colors="red", width=2)
display(to_pil_image(canvas))


## 7. Simpan hasil ke Google Drive (opsional)


In [ ]:
from google.colab import drive
drive.mount("/content/drive")
DEST=Path("/content/drive/MyDrive/fastvit_medpack")
DEST.mkdir(parents=True, exist_ok=True)
import shutil
shutil.copy2(OUT_DIR/"best.pth", DEST/"best.pth")
shutil.copy2(OUT_DIR/"history.json", DEST/"history.json")
print("Tersimpan di", DEST)
